# 深度学习环境搭建## 学习目标- 理解Conda环境管理的作用- 能创建、激活、管理Conda环境- 成功安装PyTorch并验证GPU可用- 能使用Jupyter Lab进行交互式开发- **掌握SSH端口转发，在本地浏览器中使用服务器上的Jupyter**- 了解Docker的基本概念和使用方式## 本节说明环境搭建是深度学习中最令人头疼的环节之一。本节课的目标是让每个人都能成功运行以下代码：```import torchprint(torch.cuda.is_available())  # True```

## 1. Conda环境管理### 1.1 为什么需要虚拟环境？想象一下：- 项目A需要numpy 1.21，项目B需要numpy 1.24- 项目A需要Python 3.9，项目B需要Python 3.11如果没有虚拟环境，这些冲突会让你抓狂。虚拟环境就是给每个项目一个独立的"工具箱"。**类比**：虚拟环境就像实验室里的不同实验台，每个实验台有独立的仪器和试剂，互不干扰。

### 1.2 Conda基本操作```# 创建环境conda create -n lab-training python=3.10# 激活环境conda activate lab-training# 退出环境conda deactivate# 列出所有环境conda env list# 删除环境conda env remove -n lab-training# 从配置文件创建环境conda env create -f environment.yml# 导出当前环境conda env export > environment.yml```

### 1.3 安装包```# 安装单个包conda install numpy# 安装多个包conda install numpy scipy matplotlib# 用pip安装（conda找不到的包）pip install pesq pystoi# 查看已安装的包conda list# 更新包conda update numpy```**重要规则**：- 优先用 `conda install`，因为conda会处理二进制依赖- 如果conda找不到，再用 `pip install`- 不要在同一个环境中混用conda和pip安装同一个包

### 练习1：创建Conda环境在终端中完成：1. 创建名为 `lab-training` 的环境，Python版本3.10：   ```   conda create -n lab-training python=3.10   ```2. 激活环境：   ```   conda activate lab-training   ```3. 确认Python版本：   ```   python --version   ```4. 确认你在正确的环境中：   ```   which python   ```   输出应该包含 `lab-training` 的路径

## 2. PyTorch安装与验证### 2.1 安装PyTorch安装PyTorch最安全的方式是从官网获取安装命令。根据你的系统和CUDA版本选择对应的命令：```# CUDA 11.8（最常用）conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia# CUDA 12.1conda install pytorch torchvision torchaudio pytorch-cuda=12.1 -c pytorch -c nvidia# CPU版本（没有GPU时使用）conda install pytorch torchvision torchaudio cpuonly -c pytorch```> 安装前确认服务器的CUDA版本！运行 `nvidia-smi` 查看右上角的CUDA版本。

### 2.2 验证安装安装完成后，运行以下代码验证。这是本节课最重要的验证环节。

In [ ]:
# ====== 环境验证 ======# 如果所有输出都正常，说明环境配置成功！import sysprint('Python版本:', sys.version)print('Python路径:', sys.executable)print()

In [ ]:
import torchprint('PyTorch版本:', torch.__version__)print('CUDA可用:', torch.cuda.is_available())if torch.cuda.is_available():    print('CUDA版本:', torch.version.cuda)    print('GPU数量:', torch.cuda.device_count())    print('GPU名称:', torch.cuda.get_device_name(0))    print('GPU内存: %.1f GB' % (torch.cuda.get_device_properties(0).total_mem / 1024**3))else:    print('CUDA不可用！请检查安装。')print()

In [ ]:
import torchaudioprint('torchaudio版本:', torchaudio.__version__)print()import numpy as npimport matplotlibimport scipytry:    import librosa    print('librosa版本:', librosa.__version__)except ImportError:    print('librosa未安装，可用 pip install librosa 安装')print()print('所有库导入成功！')

### 练习2：完成环境验证确保以上所有验证cell的输出都是正常的。特别注意：- `torch.cuda.is_available()` 返回 `True`- `torchaudio` 可以正常导入- `librosa` 可以正常导入如果任何一步失败，请参考本notebook末尾的"环境问题排查清单"。

## 3. pip vs conda、requirements.txt### 3.1 pip vs conda| 特性 | conda | pip ||------|-------|-----|| 包来源 | Anaconda仓库 | PyPI || 二进制依赖 | 自动处理 | 不处理 || 非Python包 | 可以安装 | 不可以 || 速度 | 较慢 | 较快 || 包数量 | 较少 | 非常多 |**规则**：优先用conda，conda找不到再用pip。### 3.2 environment.yml```name: lab-trainingchannels:  - pytorch  - conda-forge  - defaultsdependencies:  - python=3.10  - numpy  - scipy  - matplotlib  - pytorch>=2.0  - torchaudio>=2.0  - librosa  - pip  - pip:    - pesq    - pystoi```**environment.yml 更适合分享**——它包含了完整的环境定义，别人可以一键复现。

## 4. Docker基础### 4.1 Docker是什么？Docker是一种容器化技术，可以把整个环境（操作系统+依赖+代码）打包成一个"镜像"。**类比**：- Conda环境 = 在自己的房间里布置家具- Docker镜像 = 把整个房间打包，搬到任何地方都能用### 4.2 Docker基本使用```# 运行容器（带GPU支持）docker run --gpus all -p 8888:8888 -v $(pwd):/workspace lab-training# 参数说明：# --gpus all     使用所有GPU# -p 8888:8888   端口映射（Jupyter）# -v $(pwd):/workspace  挂载当前目录到容器的/workspace```你不需要会写Dockerfile——培训者已经准备好了预配置的镜像，你只需要会用就行。

## 5. SSH端口转发：在本地使用服务器上的Jupyter### 5.1 问题：服务器没有显示器我们租用的GPU服务器运行Linux，没有图形界面。Jupyter Lab是一个网页应用，需要用浏览器打开。但服务器的网页你无法直接看到——除非你把服务器的端口"转发"到你自己的电脑上。**类比**：- 服务器的Jupyter像一个在远程房间里播放的广播- SSH端口转发就像拉一根线，把这个广播接到你自己的耳机上- 你在自己的电脑浏览器中就能看到服务器上的Jupyter界面### 5.2 完整步骤**整个流程分三步**：```┌─────────────┐         SSH端口转发          ┌─────────────┐│  你的电脑    │  ←────── 隧道 ──────→       │  远程服务器  ││  (本地浏览器)│  localhost:8888 → 服务器:8888 │  Jupyter Lab │└─────────────┘                              └─────────────┘```#### 第一步：在服务器上启动Jupyter LabSSH连接到服务器后，在服务器终端运行：```# 激活环境conda activate lab-training# 启动Jupyter Labjupyter lab --no-browser --port=8888```参数说明：- `--no-browser`：服务器没有浏览器，不加这个参数会报错- `--port=8888`：使用8888端口（也可以换成其他数字，如8889、8890）启动后，终端会显示类似这样的信息：```[I 2024-01-15 10:30:00 ServerApp] jupyterlab | extension was successfully linked.[I 2024-01-15 10:30:00 ServerApp] Serving notebooks from local directory: /home/user[I 2024-01-15 10:30:00 ServerApp] Jupyter Server is running at:[I 2024-01-15 10:30:00 ServerApp] http://localhost:8888/lab?token=abc123def456```> **注意**：记下这个token（abc123def456），后面要用。不要关闭这个终端！#### 第二步：在本地电脑建立SSH隧道**打开一个新的本地终端**（不要关闭刚才的SSH连接），运行：```# 基本格式ssh -L 本地端口:localhost:远程端口 用户名@服务器地址# 实际例子ssh -L 8888:localhost:8888 student@192.168.1.100```参数说明：- `-L 8888:localhost:8888`：将本地的8888端口转发到远程服务器的localhost:8888- 你需要填入实际的用户名和服务器IP> **如果8888端口被占用**：换成其他端口，如 `ssh -L 8889:localhost:8888 ...`#### 第三步：在本地浏览器中打开Jupyter在本地电脑的浏览器地址栏中输入：```http://localhost:8888/lab?token=abc123def456```把token替换成第一步中显示的实际token。如果一切正常，你应该能看到Jupyter Lab的界面！### 5.3 一步到位：启动时自动转发更方便的方式是在SSH连接时就设置好端口转发，不需要开两个终端：```# 连接服务器的同时设置端口转发ssh -L 8888:localhost:8888 student@192.168.1.100# 连上后在服务器上启动Jupyterjupyter lab --no-browser --port=8888```这样你在本地浏览器中直接打开 `http://localhost:8888/lab?token=xxx` 即可。### 5.4 多人共用服务器的端口分配如果多个学生共用一台服务器，每人需要使用不同的端口：| 学生 | SSH命令 | Jupyter启动命令 | 浏览器地址 ||------|---------|----------------|-----------|| 学生A | `ssh -L 8888:localhost:8888 ...` | `jupyter lab --port=8888` | `localhost:8888` || 学生B | `ssh -L 8889:localhost:8889 ...` | `jupyter lab --port=8889` | `localhost:8889` || 学生C | `ssh -L 8890:localhost:8890 ...` | `jupyter lab --port=8890` | `localhost:8890` |> **重要**：如果两个人用了同一个端口，Jupyter会启动失败。请确认端口不冲突！### 5.5 VS Code Remote方式（替代方案）如果你使用VS Code，可以更方便地使用远程Jupyter：1. 安装VS Code的"Remote - SSH"扩展2. 在VS Code中按 `Ctrl+Shift+P`，输入"Remote-SSH: Connect to Host"3. 输入服务器地址（如 `student@192.168.1.100`）4. 连接后在远程服务器上打开终端，启动Jupyter5. VS Code会自动处理端口转发> **推荐**：VS Code Remote对新手最友好，但需要先学会基本的SSH操作。

### 练习3：SSH端口转发实操按以下步骤完成（约10分钟）：1. SSH连接到服务器2. 在服务器上启动Jupyter Lab（`jupyter lab --no-browser --port=8888`）3. 在本地浏览器中打开（如果直接访问不了，设置SSH端口转发）4. 在Jupyter中新建一个notebook，运行 `print("Hello from server!")`5. 确认你能看到输出**常见问题**：- 浏览器打不开 → 检查SSH端口转发是否成功- 提示token错误 → 复制服务器终端中显示的完整URL（包含token）- 端口被占用 → 换一个端口号

## 6. Jupyter Lab快捷键| 快捷键 | 功能 ||--------|------|| `Shift+Enter` | 运行当前cell，跳到下一个 || `Ctrl+Enter` | 运行当前cell，不跳转 || `A` | 在当前cell上方插入新cell || `B` | 在当前cell下方插入新cell || `DD` | 删除当前cell || `M` | 将cell切换为Markdown || `Y` | 将cell切换为Code || `Tab` | 自动补全 || `Shift+Tab` | 查看函数文档 |

## 7. 环境问题排查清单| 问题 | 可能原因 | 解决方案 ||------|---------|----------|| `conda: command not found` | Conda未安装或未加入PATH | 安装Anaconda/Miniconda，重启终端 || `torch.cuda.is_available() = False` | CUDA版本不匹配 | 运行 `nvidia-smi` 检查CUDA版本，重新安装对应版本 || `ImportError: No module named 'xxx'` | 包未安装 | `conda install xxx` 或 `pip install xxx` || `OSError: libcublas.so.11` | CUDA库缺失 | 安装对应的CUDA toolkit || Jupyter打不开 | 端口未转发 | 检查SSH端口转发配置，参考§5 || 浏览器显示"无法访问" | SSH隧道未建立 | 重新运行 `ssh -L 8888:localhost:8888 ...` || 端口被占用 | 其他人用了同一端口 | 换一个端口号 || `permission denied` | 权限不足 | 检查文件权限，`chmod +x` 或用sudo || conda安装太慢 | 默认源在国外 | 配置国内镜像源 |

In [ ]:
# ====== 一键环境检测脚本 ======# 运行此cell，如果全部显示OK，说明环境配置成功import sysprint('='*50)print('环境检测')print('='*50)# Python版本print()py_ok = sys.version_info >= (3, 10)print('Python: %s %s' % (sys.version.split()[0], 'OK' if py_ok else '(建议升级到3.10+)'))# 核心库libs = ['numpy', 'torch', 'torchaudio', 'matplotlib', 'scipy', 'librosa', 'soundfile']for lib_name in libs:    try:        mod = __import__(lib_name)        ver = getattr(mod, '__version__', 'unknown')        print('%s: %s OK' % (lib_name, ver))    except ImportError:        print('%s: 未安装' % lib_name)# GPUimport torchif torch.cuda.is_available():    print()    print('GPU: %s OK' % torch.cuda.get_device_name(0))    print('CUDA: %s OK' % torch.version.cuda)    print('GPU内存: %.1f GB OK' % (torch.cuda.get_device_properties(0).total_mem / 1024**3))else:    print()    print('GPU: 不可用 (将使用CPU，训练会较慢)')print()print('='*50)print('如果所有项都显示OK，环境配置成功！')print('='*50)

---## 本节要点1. **Conda环境** = 独立的工具箱，一个项目一个环境，互不干扰2. **安装PyTorch**时一定要匹配CUDA版本，用 `nvidia-smi` 先确认3. **优先conda安装，pip补充**——conda处理二进制依赖更好4. **Docker**是环境的终极兜底方案——一条命令启动完整环境5. **SSH端口转发**是使用远程Jupyter的关键：`ssh -L 8888:localhost:8888 user@server`6. 环境问题不可怕——大部分是版本不匹配，参考排查清单逐项检查